# Eval-0-100 Score Analysis

Judge-model (Qwen2.5-7B) scored outputs (0–100).  
Covers MedGemma / Lingshu × SLAKE / VQA-RAD, all emotion conditions, single & multi turn.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Configuration ──────────────────────────────────────────────────────────
MODEL      = "medgemma"   # "medgemma" | "lingshu"
DATASET    = "SLAKE"      # "SLAKE"    | "vqa-rad"
ONLY_CLOSED = False       # True = _closed files only; False = all questions

DIR_MAP = {
    "medgemma": Path("/workspace/EmoMedicalVLM/output/phase_2/MedGemma"),
    "lingshu":  Path("/workspace/EmoMedicalVLM/output/phase_2/Lingshu"),
}
INPUT_DIR = DIR_MAP[MODEL]

MODEL_TITLE = MODEL.capitalize()

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

In [ ]:
label_map = {
    "default":                              "default",
    "direct_clinician_neutral":             "clin_neutral_dir",
    "direct_clinician_fear_anxiety":        "clin_fear_dir",
    "direct_clinician_anger_frustration":   "clin_anger_dir",
    "direct_clinician_sadness_distress":    "clin_sad_dir",
    "direct_patient_neutral":               "pat_neutral_dir",
    "direct_patient_fear_anxiety":          "pat_fear_dir",
    "direct_patient_anger_frustration":     "pat_anger_dir",
    "direct_patient_sadness_distress":      "pat_sad_dir",
    "indirect_clinician_neutral":           "clin_neutral_Indir",
    "indirect_clinician_fear_anxiety":      "clin_fear_Indir",
    "indirect_clinician_anger_frustration": "clin_anger_Indir",
    "indirect_clinician_sadness_distress":  "clin_sad_Indir",
    "indirect_patient_neutral":             "pat_neutral_Indir",
    "indirect_patient_fear_anxiety":        "pat_fear_Indir",
    "indirect_patient_anger_frustration":   "pat_anger_Indir",
    "indirect_patient_sadness_distress":    "pat_sad_Indir",
}

row_order = list(label_map.values())

EVAL_LABELS   = ["Correct", "Partially Correct", "Incorrect", "Irrelevant", "Unknown"]
EVAL_COLORS   = ["#54A24B", "#EECA3B", "#E45756", "#9D755D", "#BAB0AC"]
EVAL_PALETTE  = dict(zip(EVAL_LABELS, EVAL_COLORS))


def parse_score(raw):
    if raw is None:
        return None
    m = re.search(r"Score:\s*(\d+)", str(raw))
    if m:
        return max(0, min(100, int(m.group(1))))
    nums = re.findall(r"\d+", str(raw))
    return max(0, min(100, int(nums[0]))) if nums else None


def score_to_label(score):
    if score is None:   return "Unknown"
    if score == 100:    return "Correct"
    if score >= 60:     return "Partially Correct"
    if score >= 1:      return "Incorrect"
    return "Irrelevant"


def build_similarity_matrix(answer_df, columns):
    arr = answer_df[columns].values
    m = arr.shape[0]
    if m == 0:
        return pd.DataFrame(np.zeros((len(columns), len(columns))), index=columns, columns=columns)
    sim = (arr[:, :, None] == arr[:, None, :]).mean(axis=0)
    return pd.DataFrame(sim, index=columns, columns=columns)


def plot_heatmap(mat, title, figsize=(12, 10), fontsize=8):
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(mat.astype(float), aspect="auto", vmin=0, vmax=1)
    fig.colorbar(im, ax=ax, label="Similarity")
    ax.set_xticks(range(len(mat.columns))); ax.set_xticklabels(mat.columns, rotation=90)
    ax.set_yticks(range(len(mat.index)));   ax.set_yticklabels(mat.index)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f"{mat.iloc[i, j]:.2f}", ha="center", va="center", fontsize=fontsize)
    ax.set_title(title)
    plt.tight_layout(); plt.show()


def safe_get(df, prompt, col):
    rows = df.loc[df["prompt"] == prompt, col]
    return float(rows.iloc[0]) if not rows.empty and not pd.isna(rows.iloc[0]) else np.nan

## 1. Load & parse files

In [ ]:
# Pattern: eval_0_100_{model}_{DATASET}_{conv_mode}_{emotion}[_closed].jsonl
PATTERN = re.compile(
    r"^eval_0_100_"
    + re.escape(MODEL) + r"_"
    + re.escape(DATASET) + r"_"
    r"(single|multi)_"
    r"(.+?)"
    r"(_closed)?\.jsonl$",
    re.IGNORECASE,
)

records   = []          # aggregated row per (conv_mode, emotion)
sample_df_parts = []    # all raw samples for per-sample analysis
answer_cols_single = {}  # prompt_short -> list of eval labels (for similarity)
answer_cols_multi  = {}

for fpath in sorted(INPUT_DIR.glob("eval_0_100_*.jsonl")):
    m = PATTERN.match(fpath.name)
    if not m:
        continue
    conv_mode   = m.group(1).lower()
    emotion_key = m.group(2).lower()
    is_closed   = m.group(3) is not None

    if ONLY_CLOSED and not is_closed:
        continue
    if not ONLY_CLOSED and is_closed:
        continue

    short = label_map.get(emotion_key)
    if short is None:
        continue

    samples = []
    with open(fpath, encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            score = item.get("score")
            if score is None:
                score = parse_score(item.get("raw_judge_response", ""))
            label = item.get("evaluation") or score_to_label(score)
            item["_score"]  = score
            item["_label"]  = label
            item["_prompt"] = short
            item["_conv"]   = conv_mode
            samples.append(item)

    if not samples:
        continue

    scores = [s["_score"] for s in samples if s["_score"] is not None]
    labels = [s["_label"] for s in samples]
    label_counts = {lbl: labels.count(lbl) for lbl in EVAL_LABELS}

    records.append({
        "conv_mode":    conv_mode,
        "emotion":      emotion_key,
        "prompt":       short,
        "mean_score":   round(float(np.mean(scores)), 2) if scores else np.nan,
        "median_score": round(float(np.median(scores)), 2) if scores else np.nan,
        "total":        len(samples),
        **{f"n_{k.replace(' ','_').lower()}": v for k, v in label_counts.items()},
    })

    sample_df_parts.extend(samples)

    if conv_mode == "single":
        answer_cols_single[short] = labels
    else:
        answer_cols_multi[short] = labels

result_df = pd.DataFrame(records)
result_df["pct_correct"] = (result_df["n_correct"] / result_df["total"] * 100).round(1)

# wide table: mean_score & pct_correct per prompt × conv_mode
wide = result_df.pivot(index="prompt", columns="conv_mode", values=["mean_score", "pct_correct"])
wide.columns = [f"{c}_{m}" for c, m in wide.columns]
wide = wide.reset_index()
avail = [p for p in row_order if p in set(wide["prompt"])]
wide["prompt"] = pd.Categorical(wide["prompt"], categories=avail, ordered=True)
wide = wide.sort_values("prompt").reset_index(drop=True)
wide["prompt"] = wide["prompt"].astype(str)

all_samples = pd.DataFrame(sample_df_parts)

print(f"Loaded {len(result_df)} conditions ({result_df['conv_mode'].nunique()} conv modes) | "
      f"samples/file: {result_df['total'].iloc[0]}")
wide

## 2. Mean score by prompt condition

In [ ]:
x = np.arange(len(wide))
w = 0.38

fig, ax = plt.subplots(figsize=(14, 5))
if "mean_score_single" in wide.columns:
    ax.bar(x - w/2, wide["mean_score_single"].fillna(0), width=w,
           label="single", color="#4C78A8", edgecolor="black")
if "mean_score_multi" in wide.columns:
    ax.bar(x + w/2, wide["mean_score_multi"].fillna(0), width=w,
           label="multi", color="#F58518", edgecolor="black")

ax.set_xticks(x); ax.set_xticklabels(wide["prompt"], rotation=90)
ax.set_ylabel("Mean Judge Score (0–100)")
ax.set_ylim(0, 110)
ax.set_title(f"{MODEL_TITLE} Mean Score by Prompt ({DATASET}, {'Closed' if ONLY_CLOSED else 'All'})")
ax.legend()
plt.tight_layout(); plt.show()

## 3. Evaluation label distribution (stacked bar)

In [ ]:
for mode in ["single", "multi"]:
    sub = result_df[result_df["conv_mode"] == mode].copy()
    if sub.empty:
        continue
    sub["prompt"] = pd.Categorical(sub["prompt"], categories=avail, ordered=True)
    sub = sub.sort_values("prompt").reset_index(drop=True)

    label_cols = {lbl: f"n_{lbl.replace(' ','_').lower()}" for lbl in EVAL_LABELS
                  if f"n_{lbl.replace(' ','_').lower()}" in sub.columns}

    totals = sub["total"].values
    bottom = np.zeros(len(sub))
    x = np.arange(len(sub))

    fig, ax = plt.subplots(figsize=(14, 5))
    for lbl, col in label_cols.items():
        pcts = sub[col].values / totals * 100
        ax.bar(x, pcts, bottom=bottom, color=EVAL_PALETTE.get(lbl, "#ccc"),
               label=lbl, edgecolor="white", linewidth=0.3)
        bottom += pcts

    ax.set_xticks(x); ax.set_xticklabels(sub["prompt"].astype(str), rotation=90)
    ax.set_ylabel("Percentage (%)")
    ax.set_ylim(0, 105)
    ax.set_title(f"{MODEL_TITLE} Evaluation Label Distribution — {mode.upper()} ({DATASET}, {'Closed' if ONLY_CLOSED else 'All'})")
    handles = [mpatches.Patch(color=EVAL_COLORS[i], label=EVAL_LABELS[i]) for i in range(len(EVAL_LABELS))]
    ax.legend(handles=handles, loc="upper right", fontsize=8)
    plt.tight_layout(); plt.show()

## 4. Score distribution (violin / box) — single vs multi

In [ ]:
if "_score" in all_samples.columns and "_conv" in all_samples.columns:
    plot_data = all_samples[["_prompt", "_conv", "_score"]].dropna(subset=["_score"])
    plot_data["_prompt"] = pd.Categorical(plot_data["_prompt"], categories=avail, ordered=True)
    plot_data = plot_data.sort_values("_prompt")

    prompts = [p for p in avail if p in plot_data["_prompt"].values]
    x = np.arange(len(prompts))
    w = 0.35

    fig, ax = plt.subplots(figsize=(14, 5))
    for offset, mode, color in [(-w/2, "single", "#4C78A8"), (w/2, "multi", "#F58518")]:
        sub = plot_data[plot_data["_conv"] == mode]
        if sub.empty:
            continue
        medians = [sub.loc[sub["_prompt"] == p, "_score"].median() for p in prompts]
        ax.bar(x + offset, medians, width=w, label=f"{mode} (median)",
               color=color, edgecolor="black", alpha=0.85)

    ax.set_xticks(x); ax.set_xticklabels(prompts, rotation=90)
    ax.set_ylabel("Median Score")
    ax.set_ylim(0, 110)
    ax.set_title(f"{MODEL_TITLE} Median Score by Prompt ({DATASET})")
    ax.legend()
    plt.tight_layout(); plt.show()

## 5. Similarity matrix — evaluation labels across conditions

In [ ]:
for mode, cols_dict in [("single", answer_cols_single), ("multi", answer_cols_multi)]:
    if not cols_dict:
        continue
    ordered = [p for p in avail if p in cols_dict]
    min_len = min(len(v) for v in cols_dict.values())
    sim_input = pd.DataFrame({k: cols_dict[k][:min_len] for k in ordered})
    sim_mat = build_similarity_matrix(sim_input, ordered)
    plot_heatmap(
        sim_mat,
        title=f"{MODEL_TITLE} Evaluation-Label Similarity ({DATASET}, {mode.upper()})",
    )

## 6. Role effect (clinician vs patient)

In [ ]:
emotions = ["neutral", "fear", "anger", "sad"]
styles   = ["dir", "Indir"]

role_rows = []
for emo in emotions:
    for style in styles:
        pc = f"clin_{emo}_{style}"
        pp = f"pat_{emo}_{style}"
        for mode in ["single", "multi"]:
            col = f"mean_score_{mode}"
            if col not in wide.columns:
                continue
            vc = safe_get(wide, pc, col)
            vp = safe_get(wide, pp, col)
            role_rows.append({
                "condition": f"{emo}_{style}",
                "conv_mode": mode,
                "clin_score": vc,
                "pat_score":  vp,
                "gap":        abs(vc - vp) if not (np.isnan(vc) or np.isnan(vp)) else np.nan,
            })

role_df = pd.DataFrame(role_rows)

for mode in ["single", "multi"]:
    sub = role_df[role_df["conv_mode"] == mode].set_index("condition")
    if sub.empty:
        continue
    x = np.arange(len(sub))
    w = 0.38
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - w/2, sub["clin_score"], width=w, label="Clinician", color="#4C78A8", edgecolor="black")
    ax.bar(x + w/2, sub["pat_score"],  width=w, label="Patient",   color="#F58518", edgecolor="black")
    ax.set_xticks(x); ax.set_xticklabels(sub.index, rotation=45, ha="right")
    ax.set_ylabel("Mean Score")
    ax.set_ylim(0, 110)
    ax.set_title(f"{MODEL_TITLE} Role Effect — {mode.upper()} ({DATASET})")
    ax.legend()
    plt.tight_layout(); plt.show()

## 7. Direct vs indirect comparison

In [ ]:
di_rows = []
for mode in ["single", "multi"]:
    col = f"mean_score_{mode}"
    if col not in wide.columns:
        continue
    for role in ["clin", "pat"]:
        for emo in emotions:
            pd_ = f"{role}_{emo}_dir"
            pi  = f"{role}_{emo}_Indir"
            vd  = safe_get(wide, pd_, col)
            vi  = safe_get(wide, pi,  col)
            di_rows.append({
                "condition":  f"{role}_{emo}",
                "conv_mode":  mode,
                "direct":     vd,
                "indirect":   vi,
                "indir_minus_dir": vi - vd if not (np.isnan(vd) or np.isnan(vi)) else np.nan,
            })

di_df = pd.DataFrame(di_rows)

for mode in ["single", "multi"]:
    sub = di_df[di_df["conv_mode"] == mode].set_index("condition")
    if sub.empty:
        continue
    x = np.arange(len(sub))
    w = 0.38
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - w/2, sub["direct"],   width=w, label="Direct",   color="#54A24B", edgecolor="black")
    ax.bar(x + w/2, sub["indirect"], width=w, label="Indirect", color="#72B7B2", edgecolor="black")
    ax.set_xticks(x); ax.set_xticklabels(sub.index, rotation=45, ha="right")
    ax.set_ylabel("Mean Score")
    ax.set_ylim(0, 110)
    ax.set_title(f"{MODEL_TITLE} Direct vs Indirect — {mode.upper()} ({DATASET})")
    ax.legend()
    plt.tight_layout(); plt.show()

di_df.round(2)

## 8. Metadata breakdown — modality & content type

In [ ]:
if "modality" in all_samples.columns and "_score" in all_samples.columns:
    for meta_col in ["modality", "content_type", "answer_type"]:
        if meta_col not in all_samples.columns:
            continue
        grp = (
            all_samples.dropna(subset=["_score", meta_col])
            .groupby([meta_col, "_conv"])["_score"]
            .mean()
            .unstack("_conv")
            .round(2)
        )
        if grp.empty:
            continue
        ax = grp.plot(kind="bar", figsize=(10, 4), color=["#4C78A8", "#F58518"],
                      edgecolor="black")
        ax.set_ylabel("Mean Score")
        ax.set_ylim(0, 110)
        ax.set_title(f"{MODEL_TITLE} Mean Score by {meta_col.replace('_',' ').title()} ({DATASET})")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
        plt.tight_layout(); plt.show()

## 9. Cross-model comparison (MedGemma vs Lingshu)

In [ ]:
other_model = "lingshu" if MODEL == "medgemma" else "medgemma"
other_dir   = DIR_MAP[other_model]

other_records = []
for fpath in sorted(other_dir.glob("eval_0_100_*.jsonl")):
    m = re.match(
        r"^eval_0_100_" + re.escape(other_model) + r"_"
        + re.escape(DATASET) + r"_(single|multi)_(.+?)(_closed)?\.jsonl$",
        fpath.name, re.IGNORECASE,
    )
    if not m:
        continue
    conv_mode   = m.group(1).lower()
    emotion_key = m.group(2).lower()
    is_closed   = m.group(3) is not None
    if ONLY_CLOSED != is_closed:
        continue
    short = label_map.get(emotion_key)
    if short is None:
        continue
    scores = []
    with open(fpath, encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            s = item.get("score")
            if s is None:
                s = parse_score(item.get("raw_judge_response", ""))
            if s is not None:
                scores.append(s)
    if scores:
        other_records.append({
            "conv_mode": conv_mode,
            "prompt":    short,
            "mean_score": round(float(np.mean(scores)), 2),
        })

other_df = pd.DataFrame(other_records)

for mode in ["single", "multi"]:
    cur  = result_df[result_df["conv_mode"] == mode][["prompt", "mean_score"]].rename(columns={"mean_score": MODEL})
    oth  = other_df[other_df["conv_mode"] == mode][["prompt", "mean_score"]].rename(columns={"mean_score": other_model})
    if cur.empty or oth.empty:
        continue
    merged = cur.merge(oth, on="prompt", how="outer")
    merged["prompt"] = pd.Categorical(merged["prompt"], categories=avail, ordered=True)
    merged = merged.sort_values("prompt").reset_index(drop=True)

    x = np.arange(len(merged))
    w = 0.38
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - w/2, merged[MODEL].fillna(0),       width=w, label=MODEL,       color="#4C78A8", edgecolor="black")
    ax.bar(x + w/2, merged[other_model].fillna(0), width=w, label=other_model, color="#E45756", edgecolor="black")
    ax.set_xticks(x); ax.set_xticklabels(merged["prompt"].astype(str), rotation=90)
    ax.set_ylabel("Mean Score")
    ax.set_ylim(0, 110)
    ax.set_title(f"Cross-Model Comparison — {mode.upper()} ({DATASET}, {'Closed' if ONLY_CLOSED else 'All'})")
    ax.legend()
    plt.tight_layout(); plt.show()

## 10. Summary table

In [ ]:
summary_rows = []
for mode in ["single", "multi"]:
    sub = result_df[result_df["conv_mode"] == mode]
    if sub.empty:
        continue
    summary_rows.append({
        "model":      MODEL,
        "dataset":    DATASET,
        "conv_mode":  mode,
        "mean_score": round(sub["mean_score"].mean(), 2),
        "pct_correct":round(sub["pct_correct"].mean(), 1),
        "n_conditions": len(sub),
    })

    # add other model if loaded
    oth = other_df[other_df["conv_mode"] == mode]
    if not oth.empty:
        summary_rows.append({
            "model":      other_model,
            "dataset":    DATASET,
            "conv_mode":  mode,
            "mean_score": round(oth["mean_score"].mean(), 2),
            "pct_correct": np.nan,
            "n_conditions": len(oth),
        })

pd.DataFrame(summary_rows)